# Job Task: Ingest and Validate

> **This notebook runs as a Databricks Job task.** Participants trigger it from `02_trigger_retrain.ipynb`.

Task 1 of the retrain pipeline:
- Load data from the shared source table
- Run data quality checks
- Fail the job if checks do not pass (prevents training on bad data)

In [0]:
dbutils.widgets.text("catalog", "workshop", "Catalog")
dbutils.widgets.text("schema", "00_shared", "Schema")
dbutils.widgets.text("config_path", "config.yml", "Config Path")

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")
print(f"catalog={catalog}, schema={schema}")

In [0]:
%pip install /Volumes/{catalog}/00_shared/wheels/churn_model-0.1.0-py3-none-any.whl --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import yaml

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")
config_path  = dbutils.widgets.get("config_path")

with open(config_path) as f:
    config = yaml.safe_load(f)

from churn_model.features import run_data_quality_checks
df = spark.table(f"{catalog}.`00_shared`.telco_churn").toPandas()
results = run_data_quality_checks(df, config)

failed = results[~results["passed"]]
print(f"Data quality: {len(results) - len(failed)}/{len(results)} checks passed")

if len(failed) > 0:
    print("FAILED CHECKS:")
    print(failed[["check_name", "actual_value", "details"]].to_string())
    raise ValueError(f"Data quality checks failed: {failed['check_name'].tolist()}")

print("✓ All data quality checks passed. Proceeding to training.")